# **DejaVu + prefetch (pageable, pinned) and profiling**

Cells copied unchanged (outputs kept) from `archive/OS_MAIN.ipynb`: original cell indices [0, 1, 6, 7, 15, 16, 17, 18, 19, 20, 21, 25, 26, 27, 28].

In [1]:
import torch
import time
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, DynamicCache
import transformers

transformers.utils.logging.disable_progress_bar()

torch.cuda.set_per_process_memory_fraction(0.45)

# model_name = "gpt2"
model_name = "facebook/opt-6.7b"

device = "cuda"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    attn_implementation="eager"
).to(device)
model.eval()

print(f"Model: {model_name}")
print(f"Device: {torch.cuda.get_device_name()}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB")

Model: facebook/opt-6.7b
Device: NVIDIA A100-SXM4-40GB
VRAM: 39.5 GB


In [2]:
from datasets import load_dataset

# WikiText-2
dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")

# Get long articles by filtering for substantial text
long_prompts = []
for item in dataset:
    text = item["text"].strip()
    tokens = tokenizer.encode(text)
    if len(tokens) > 200:  # only keep long articles
        long_prompts.append(text)
    if len(long_prompts) >= 32:
        break

print(f"Found {len(long_prompts)}  prompts")
for i, p in enumerate(long_prompts[:4]):
    tokens = tokenizer.encode(p)
    print(f"Prompt {i}: {len(tokens)} tokens | {p[:80]}...")

Found 32  prompts
Prompt 0: 217 tokens | He had a recurring role in 2003 on two episodes of The Bill , as character " Con...
Prompt 1: 205 tokens | In 2006 Boulter starred in the play Citizenship written by Mark Ravenhill . The ...
Prompt 2: 246 tokens | Boulter starred in two films in 2008 , Daylight Robbery by filmmaker Paris Leont...
Prompt 3: 268 tokens | Du Fu 's work is notable above all for its range . Chinese critics traditionally...


**Helper Functions**

In [2]:
def get_kv_layers(kv_cache):
    """Convert DynamicCache or other cache formats into standard ((k,v),(k,v),...) format."""
    if kv_cache is None:
        return None
    # Newer transformers: DynamicCache stores tensors in .layers[i].keys / .layers[i].values
    if hasattr(kv_cache, "layers") and isinstance(kv_cache.layers, list):
        return [(layer.keys, layer.values) for layer in kv_cache.layers]
    # Older transformers: DynamicCache stores tensors in .key_cache / .value_cache
    if hasattr(kv_cache, "key_cache") and hasattr(kv_cache, "value_cache"):
        return list(zip(kv_cache.key_cache, kv_cache.value_cache))
    # Try legacy tuple conversion as final fallback
    if hasattr(kv_cache, "to_legacy_cache"):
        legacy = kv_cache.to_legacy_cache()
        if isinstance(legacy, (tuple, list)):
            return legacy
    if isinstance(kv_cache, (tuple, list)):
        return kv_cache
    raise TypeError(f"Unsupported KV cache type: {type(kv_cache)}")

def get_cache_seq_len(kv_cache):
    """Return current sequence length of KV cache."""
    layers = get_kv_layers(kv_cache)
    if layers is None or len(layers) == 0:
        return 0
    first_key = layers[0][0]
    return first_key.shape[-2]

def move_full_kv_to_device_timed(kv_cache, target_device):
    """Move KV tensors from CPU → device and reconstruct a DynamicCache the model can consume."""
    start = time.perf_counter()
    dev_layers = [(k.to(target_device), v.to(target_device)) for k, v in kv_cache]
    # Rebuild as DynamicCache so the model receives the correct cache type
    cache = build_dynamic_cache(dev_layers)
    end = time.perf_counter()
    return cache, (end - start)

def extract_new_kv_chunk(full_kv_cache, old_seq_len):
    """Extract only newly generated KV entries from updated cache."""
    layers = get_kv_layers(full_kv_cache)
    new_layers = []
    for k, v in layers:
        new_k = k[:, :, old_seq_len:, :].detach().cpu()
        new_v = v[:, :, old_seq_len:, :].detach().cpu()
        new_layers.append((new_k, new_v))
    return tuple(new_layers)

def build_dynamic_cache(kv_layers):
  """Reconstruct a DynamicCache object from a list of (key, value) tensor pairs."""
  cache = DynamicCache()
  for layer_idx, (k, v) in enumerate(kv_layers):
    # update() expects shape (batch, heads, seq, head_dim) and stores it internally
    cache.update(k, v, layer_idx)
  return cache

def append_kv_caches(cpu_kv_cache, new_chunk_cache):
  """Append new KV chunk to CPU side KV cache."""
  merged = []
  for (old_k, old_v), (new_k, new_v) in zip(cpu_kv_cache, new_chunk_cache):
    merged_k = torch.cat([old_k, new_k], dim=-2)
    merged_v = torch.cat([old_v, new_v], dim=-2)
    merged.append((merged_k, merged_v))
  return tuple(merged)

# **DEJAVU+PREFETCH WITHOUT PINNED MEMORY**

In [ ]:
def run_dejavu_parallel_benchmark():
  microbatches_count = [4,8,16]
  n_tokens = [256]
  # microbatches_count = [2]
  # n_tokens = [32]
  results = []

  total_vram_mb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 2)

  for D in microbatches_count:
    for N in n_tokens:

      prompts = LONG_PROMPTS[:D] # Select D prompts

      torch.cuda.empty_cache()
      torch.cuda.reset_peak_memory_stats()

      total_start= time.perf_counter()

      with torch.no_grad():
        kv_caches = []
        next_tokens = []

        # PREFILL
        for i in range(D):
          inputs = tokenizer(prompts[i], return_tensors="pt").to(device)
          outputs = model(input_ids = inputs.input_ids,use_cache = True)
          kv_caches.append(outputs.past_key_values)
          next_token = torch.argmax(outputs.logits[:,-1,:],dim=1).unsqueeze(1)
          next_tokens.append(next_token)

        # Move KV cache to CPU
        cpu_kv_caches = []
        for i in range(D):
          layers = get_kv_layers(kv_caches[i])
          cpu_layers = [(k.detach().cpu(), v.detach().cpu()) for k, v in layers]
          cpu_kv_caches.append(cpu_layers)
        del kv_caches
        torch.cuda.empty_cache()

        transfer_time = 0.0

        prefetch_stream = torch.cuda.Stream()
        offload_stream = torch.cuda.Stream()
        # DECODE

        # Prefetch first microbatch before loop
        with torch.cuda.stream(prefetch_stream):
          prefetched = [(k.to(device, non_blocking=True), v.to(device, non_blocking=True)) for k, v in cpu_kv_caches[0]]

        for step in range(N-1):
          for i in range(D):
            next_i = (i + 1) % D

            prefetch_stream.synchronize()   # make sure upload is done
            offload_stream.synchronize()    # make sure previous offload is done

            # Use current prefetched data
            gpu_layers = prefetched         # this is microbatch i's data

            with torch.cuda.stream(prefetch_stream):
              prefetched = [(k.to(device, non_blocking=True),v.to(device, non_blocking=True)) for k, v in cpu_kv_caches[next_i]]

            gpu_cache = build_dynamic_cache(gpu_layers)
            old_seq_length = gpu_cache.get_seq_length()
            outputs = model(
                input_ids = next_tokens[i],
                past_key_values = gpu_cache,
                use_cache = True
              )

            # Append newly generated kv chunk to the cpu cache
            with torch.cuda.stream(offload_stream):
              new_chunk = extract_new_kv_chunk(outputs.past_key_values, old_seq_length)
              cpu_kv_caches[i] = append_kv_caches(cpu_kv_caches[i], new_chunk)

            next_tokens[i] = torch.argmax(outputs.logits[:,-1,:],dim=1).unsqueeze(1)

        try:
          del gpu_cache, new_kv_chunk
        except:
          pass
        torch.cuda.empty_cache()
      total_end = time.perf_counter()

      # METRICS
      latency = total_end - total_start
      total_tokens = D * N
      throughput = total_tokens / latency
      peak_vram_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
      vram_util = (peak_vram_mb / total_vram_mb) * 100

      total_kv_bytes = 0
      for cache in cpu_kv_caches:
        for k, v in cache:
          total_kv_bytes += k.nelement() * k.element_size()
          total_kv_bytes += v.nelement() * v.element_size()
      cpu_kv_cache_mb = total_kv_bytes / (1024 ** 2)

      print(f"D={D:<4} N={N:<4} | Latency={latency:.3f}s | Throughput={throughput:.1f} tok/s | Peak VRAM={peak_vram_mb:.1f}MB | VRAM%={vram_util:.1f}% | CPU KV Cache={cpu_kv_cache_mb:.1f}MB")

      results.append({
                "Microbatches (D)": D,
                "Tokens Generated (N)": N,
                "Latency (s)": latency,
                "Throughput (tok/s)": throughput,
                "Peak VRAM (MB)": peak_vram_mb,
                "VRAM Util (%)": vram_util,
                "CPU KV Cache Total (MB)": cpu_kv_cache_mb
            })

      # CLEANUP
      del next_tokens
      torch.cuda.empty_cache()

  return pd.DataFrame(results)



In [ ]:
run_dejavu_parallel_benchmark()

D=4    N=256  | Latency=67.042s | Throughput=15.3 tok/s | Peak VRAM=13631.3MB | VRAM%=33.7% | CPU KV Cache=639.0MB
D=8    N=256  | Latency=132.958s | Throughput=15.4 tok/s | Peak VRAM=13629.4MB | VRAM%=33.7% | CPU KV Cache=1269.0MB
D=16   N=256  | Latency=266.880s | Throughput=15.3 tok/s | Peak VRAM=13809.5MB | VRAM%=34.1% | CPU KV Cache=2537.5MB


,Microbatches (D),Tokens Generated (N),Latency (s),Throughput (tok/s),Peak VRAM (MB),VRAM Util (%),CPU KV Cache Total (MB)
0,4,256,67.042480,15.273898,13631.263184,33.706231,639.0
1,8,256,132.957923,15.403369,13629.381348,33.701578,1269.0
2,16,256,266.880122,15.347715,13809.469727,34.146885,2537.5


# **DEJAVU + PREFETCH**

In [3]:
def unpack_kv_cache(kv_cache):
    """Convert DynamicCache or other cache formats into standard ((k,v),(k,v),...) format."""
    if kv_cache is None:
        return None
    # Newest transformers: DynamicCache has .layers list of DynamicLayer objects
    if hasattr(kv_cache, "layers") and isinstance(kv_cache.layers, list):
        return [(layer.keys, layer.values) for layer in kv_cache.layers]
    # Legacy tuple format
    if isinstance(kv_cache, (tuple, list)):
        return kv_cache
    raise TypeError(f"Unsupported KV cache type: {type(kv_cache)}")


def get_seq_len(kv_cache):
    """Return current sequence length of KV cache."""
    layers = unpack_kv_cache(kv_cache)
    if layers is None or len(layers) == 0:
        return 0
    return layers[0][0].shape[-2]


def kv_cache_size_mb(kv_cache):
    """Compute total KV cache memory size in MB."""
    layers = unpack_kv_cache(kv_cache)
    if layers is None:
        return 0
    total = 0
    for k, v in layers:
        total += k.nelement() * k.element_size()
        total += v.nelement() * v.element_size()
    return total / (1024 ** 2)


def offload_kv_to_cpu_pinned(kv_cache):
    """Move KV cache to pinned CPU memory for async transfer."""
    layers = unpack_kv_cache(kv_cache)
    return tuple(
        (k.detach().contiguous().cpu().pin_memory(),
         v.detach().contiguous().cpu().pin_memory())
        for k, v in layers
    )


def extract_new_kv_chunk_pinned(full_kv_cache, old_seq_len):
    """Extract only newly generated KV entries and store in pinned CPU memory."""
    layers = unpack_kv_cache(full_kv_cache)
    return tuple(
        (k[:, :, old_seq_len:, :].detach().contiguous().cpu().pin_memory(),
         v[:, :, old_seq_len:, :].detach().contiguous().cpu().pin_memory())
        for k, v in layers
    )


def append_kv_chunk_pinned(cpu_kv_cache, new_chunk):
    """Append new KV chunk to CPU-side KV cache, keeping result in pinned memory."""
    return tuple(
        (torch.cat([old_k, new_k], dim=-2).contiguous().pin_memory(),
         torch.cat([old_v, new_v], dim=-2).contiguous().pin_memory())
        for (old_k, old_v), (new_k, new_v) in zip(cpu_kv_cache, new_chunk)
    )


def build_kv_cache(kv_layers):
    """Reconstruct a DynamicCache from a list of (key, value) tensor pairs."""
    cache = DynamicCache()
    for layer_idx, (k, v) in enumerate(kv_layers):
        cache.update(k, v, layer_idx)
    return cache

In [4]:
def run_dejavu_prefetch_benchmark():
    microbatches_count = [8,16,32]
    n_tokens = [256, 512]
    results = []

    total_vram_mb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 2)

    for D in microbatches_count:
        for N in n_tokens:

            prompts = long_prompts[:D]

            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()

            total_start = time.perf_counter()

            with torch.no_grad():

                # PREFILL
                cpu_kv_caches = []
                next_tokens = []
                for i in range(D):
                    inputs = tokenizer(prompts[i], return_tensors="pt").to(device)
                    outputs = model(input_ids=inputs.input_ids, use_cache=True)
                    next_tokens.append(torch.argmax(outputs.logits[:, -1, :], dim=1).unsqueeze(1))
                    cpu_kv_caches.append(offload_kv_to_cpu_pinned(outputs.past_key_values))
                    del outputs
                torch.cuda.empty_cache()
                torch.cuda.reset_peak_memory_stats()

                prefetch_stream = torch.cuda.Stream()
                offload_stream = torch.cuda.Stream()

                # Prefetch first microbatch before loop
                with torch.cuda.stream(prefetch_stream):
                    prefetched = [(k.to(device, non_blocking=True), v.to(device, non_blocking=True))
                                  for k, v in cpu_kv_caches[0]]

                # DECODE
                for step in range(N - 1):
                    for i in range(D):
                        next_i = (i + 1) % D

                        prefetch_stream.synchronize()
                        offload_stream.synchronize()

                        gpu_layers = prefetched

                        with torch.cuda.stream(prefetch_stream):
                            prefetched = [(k.to(device, non_blocking=True), v.to(device, non_blocking=True))
                                          for k, v in cpu_kv_caches[next_i]]

                        gpu_cache = build_kv_cache(gpu_layers)
                        old_seq_len = gpu_cache.get_seq_length()
                        outputs = model(
                            input_ids=next_tokens[i],
                            past_key_values=gpu_cache,
                            use_cache=True
                        )

                        with torch.cuda.stream(offload_stream):
                            new_chunk = extract_new_kv_chunk_pinned(outputs.past_key_values, old_seq_len)
                            cpu_kv_caches[i] = append_kv_chunk_pinned(cpu_kv_caches[i], new_chunk)

                        next_tokens[i] = torch.argmax(outputs.logits[:, -1, :], dim=1).unsqueeze(1)

                try:
                    del gpu_cache, new_chunk
                except:
                    pass
                torch.cuda.empty_cache()

            total_end = time.perf_counter()

            latency = total_end - total_start
            throughput = (D * N) / latency
            peak_vram_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
            vram_util = (peak_vram_mb / total_vram_mb) * 100
            cpu_kv_mb = sum(kv_cache_size_mb(cache) for cache in cpu_kv_caches)

            print(f"D={D:<4} N={N:<4} | Latency={latency:.3f}s | Throughput={throughput:.1f} tok/s | "
                  f"Peak VRAM={peak_vram_mb:.1f}MB | VRAM%={vram_util:.1f}% | CPU KV Cache={cpu_kv_mb:.1f}MB")

            results.append({
                "Microbatches (D)": D,
                "Tokens Generated (N)": N,
                "Latency (s)": latency,
                "Throughput (tok/s)": throughput,
                "Peak VRAM (MB)": peak_vram_mb,
                "VRAM Util (%)": vram_util,
                "CPU KV Cache Total (MB)": cpu_kv_mb
            })

            del next_tokens
            torch.cuda.empty_cache()

    return pd.DataFrame(results)

In [5]:
run_dejavu_prefetch_benchmark()

D=8    N=256  | Latency=114.670s | Throughput=17.9 tok/s | Peak VRAM=13879.3MB | VRAM%=34.3% | CPU KV Cache=2053.5MB
D=8    N=512  | Latency=243.653s | Throughput=16.8 tok/s | Peak VRAM=14373.3MB | VRAM%=35.5% | CPU KV Cache=3077.5MB
D=16   N=256  | Latency=219.008s | Throughput=18.7 tok/s | Peak VRAM=13876.1MB | VRAM%=34.3% | CPU KV Cache=4091.5MB
D=16   N=512  | Latency=493.419s | Throughput=16.6 tok/s | Peak VRAM=14376.4MB | VRAM%=35.5% | CPU KV Cache=6139.5MB
D=32   N=256  | Latency=472.454s | Throughput=17.3 tok/s | Peak VRAM=13980.9MB | VRAM%=34.6% | CPU KV Cache=8323.5MB
D=32   N=512  | Latency=1017.492s | Throughput=16.1 tok/s | Peak VRAM=14487.9MB | VRAM%=35.8% | CPU KV Cache=12419.5MB


,Microbatches (D),Tokens Generated (N),Latency (s),Throughput (tok/s),Peak VRAM (MB),VRAM Util (%),CPU KV Cache Total (MB)
0,8,256,114.670058,17.859937,13879.300781,34.319557,2053.5
1,8,512,243.653211,16.810778,14373.343262,35.541183,3077.5
2,16,256,219.008296,18.702488,13876.113281,34.311675,4091.5
3,16,512,493.418662,16.602534,14376.366699,35.548660,6139.5
4,32,256,472.454237,17.339246,13980.883789,34.570743,8323.5
5,32,512,1017.492024,16.102338,14487.910645,35.824476,12419.5


# **Profiler** - DejaVu Parallel

In [ ]:
def run_dejavu_profiled(D=16, N=512):
    prompts = LONG_PROMPTS[:D]

    import gc
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    total_vram_mb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 2)

    t_sync_prefetch = 0.0
    t_sync_offload = 0.0
    t_kick_prefetch = 0.0
    t_build_cache = 0.0
    t_forward = 0.0
    t_kick_offload = 0.0
    t_argmax = 0.0

    t_start = time.perf_counter()

    with torch.no_grad():
        kv_caches = []
        next_tokens = []
        for i in range(D):
            inputs = tokenizer(prompts[i], return_tensors="pt").to(device)
            outputs = model(input_ids=inputs.input_ids, use_cache=True)
            kv_caches.append(outputs.past_key_values)
            next_tokens.append(torch.argmax(outputs.logits[:, -1, :], dim=1).unsqueeze(1))

        cpu_kv_caches = []
        for i in range(D):
            layers = unpack_kv_cache(kv_caches[i])
            cpu_layers = tuple((k.detach().cpu(), v.detach().cpu()) for k, v in layers)
            cpu_kv_caches.append(cpu_layers)
        del kv_caches
        torch.cuda.empty_cache()

        prefetch_stream = torch.cuda.Stream()
        offload_stream = torch.cuda.Stream()

        with torch.cuda.stream(prefetch_stream):
            prefetched = tuple((k.to(device, non_blocking=True), v.to(device, non_blocking=True)) for k, v in cpu_kv_caches[0])

        t_decode_start = time.perf_counter()

        with torch.profiler.profile(
            activities=[
                torch.profiler.ProfilerActivity.CPU,
                torch.profiler.ProfilerActivity.CUDA,
            ],
            schedule=torch.profiler.schedule(
                wait=1,
                warmup=1,
                active=3,
                repeat=1
            ),
            on_trace_ready=torch.profiler.tensorboard_trace_handler('./profiler_logs/dejavu_unpinned'),
            record_shapes=True,
            with_stack=False
        ) as prof:
            for step in range(N - 1):
                for i in range(D):
                    next_i = (i + 1) % D

                    t0 = time.perf_counter()
                    prefetch_stream.synchronize()
                    t_sync_prefetch += time.perf_counter() - t0

                    t0 = time.perf_counter()
                    offload_stream.synchronize()
                    t_sync_offload += time.perf_counter() - t0

                    gpu_layers = prefetched

                    t0 = time.perf_counter()
                    with torch.cuda.stream(prefetch_stream):
                        prefetched = tuple((k.to(device, non_blocking=True), v.to(device, non_blocking=True)) for k, v in cpu_kv_caches[next_i])
                    t_kick_prefetch += time.perf_counter() - t0

                    t0 = time.perf_counter()
                    gpu_cache = build_kv_cache(gpu_layers)
                    t_build_cache += time.perf_counter() - t0

                    t0 = time.perf_counter()
                    old_seq = gpu_cache.get_seq_length()
                    with torch.profiler.record_function("forward_pass"):
                        outputs = model(input_ids=next_tokens[i], past_key_values=gpu_cache, use_cache=True)
                    torch.cuda.synchronize()
                    t_forward += time.perf_counter() - t0

                    t0 = time.perf_counter()
                    with torch.profiler.record_function("offload"):
                        with torch.cuda.stream(offload_stream):
                            new_kv = unpack_kv_cache(outputs.past_key_values)
                            new_chunk = tuple((k[:, :, old_seq:, :].detach().cpu(), v[:, :, old_seq:, :].detach().cpu()) for k, v in new_kv)
                            cpu_kv_caches[i] = tuple(
                                (torch.cat([old_k, new_k], dim=-2), torch.cat([old_v, new_v], dim=-2))
                                for (old_k, old_v), (new_k, new_v) in zip(cpu_kv_caches[i], new_chunk)
                            )
                    t_kick_offload += time.perf_counter() - t0

                    t0 = time.perf_counter()
                    next_tokens[i] = torch.argmax(outputs.logits[:, -1, :], dim=1).unsqueeze(1)
                    t_argmax += time.perf_counter() - t0

                    del gpu_layers, gpu_cache, outputs
                    prof.step()

        t_decode_end = time.perf_counter()

    t_end = time.perf_counter()
    decode_time = t_decode_end - t_decode_start
    latency = t_end - t_start
    peak_vram = torch.cuda.max_memory_allocated() / (1024 ** 2)
    cpu_kv_mb = sum(kv_cache_size_mb(cache) for cache in cpu_kv_caches)

    print(f"D={D}, N={N}, Total decode steps: {(N-1)*D}")
    print(f"Decode total:          {decode_time:.3f}s")
    print(f"\n--- Decode breakdown ---")
    print(f"Sync prefetch (wait):  {t_sync_prefetch:.3f}s  ({100*t_sync_prefetch/decode_time:.1f}%)")
    print(f"Sync offload (wait):   {t_sync_offload:.3f}s  ({100*t_sync_offload/decode_time:.1f}%)")
    print(f"Kick prefetch:         {t_kick_prefetch:.3f}s  ({100*t_kick_prefetch/decode_time:.1f}%)")
    print(f"Build cache:           {t_build_cache:.3f}s  ({100*t_build_cache/decode_time:.1f}%)")
    print(f"Forward pass:          {t_forward:.3f}s  ({100*t_forward/decode_time:.1f}%)")
    print(f"Kick offload:          {t_kick_offload:.3f}s  ({100*t_kick_offload/decode_time:.1f}%)")
    print(f"Argmax:                {t_argmax:.3f}s  ({100*t_argmax/decode_time:.1f}%)")

    print("\n--- torch.profiler top ops ---")
    print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=15))

    print(f"\nD={D:<4} N={N:<4} | Latency={latency:.3f}s | Throughput={D*N/latency:.1f} tok/s | Peak VRAM={peak_vram:.1f}MB | VRAM%={100*peak_vram/total_vram_mb:.1f}% | CPU KV={cpu_kv_mb:.1f}MB")

    del cpu_kv_caches, next_tokens, prefetched
    gc.collect()
    torch.cuda.empty_cache()

run_dejavu_profiled()

/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


D=16, N=512, Total decode steps: 8176
Decode total:          776.331s

--- Decode breakdown ---
Sync prefetch (wait):  0.176s  (0.0%)
Sync offload (wait):   0.046s  (0.0%)
Kick prefetch:         322.286s  (41.5%)
Build cache:           17.225s  (2.2%)
Forward pass:          295.710s  (38.1%)
Kick offload:          137.675s  (17.7%)
Argmax:                0.853s  (0.1%)

--- torch.profiler top ops ---
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
 

# **Profiler** - DejaVu Parellel with Pinned Memory

In [ ]:
def unpack_kv_cache(kv_cache):
    """Convert DynamicCache or other cache formats into standard ((k,v),(k,v),...) format."""
    if kv_cache is None:
        return None
    # Newest transformers: DynamicCache has .layers list of DynamicLayer objects
    if hasattr(kv_cache, "layers") and isinstance(kv_cache.layers, list):
        return [(layer.keys, layer.values) for layer in kv_cache.layers]
    # Legacy tuple format
    if isinstance(kv_cache, (tuple, list)):
        return kv_cache
    raise TypeError(f"Unsupported KV cache type: {type(kv_cache)}")


def get_seq_len(kv_cache):
    """Return current sequence length of KV cache."""
    layers = unpack_kv_cache(kv_cache)
    if layers is None or len(layers) == 0:
        return 0
    return layers[0][0].shape[-2]


def kv_cache_size_mb(kv_cache):
    """Compute total KV cache memory size in MB."""
    layers = unpack_kv_cache(kv_cache)
    if layers is None:
        return 0
    total = 0
    for k, v in layers:
        total += k.nelement() * k.element_size()
        total += v.nelement() * v.element_size()
    return total / (1024 ** 2)


def offload_kv_to_cpu(kv_cache):
    """Move KV cache to pinned CPU memory for async transfer."""
    layers = unpack_kv_cache(kv_cache)
    return tuple(
        (k.detach().contiguous().cpu().pin_memory(),
         v.detach().contiguous().cpu().pin_memory())
        for k, v in layers
    )


def extract_new_kv_chunk(full_kv_cache, old_seq_len):
    """Extract only newly generated KV entries and store in pinned CPU memory."""
    layers = unpack_kv_cache(full_kv_cache)
    return tuple(
        (k[:, :, old_seq_len:, :].detach().contiguous().cpu().pin_memory(),
         v[:, :, old_seq_len:, :].detach().contiguous().cpu().pin_memory())
        for k, v in layers
    )


def append_kv_chunk(cpu_kv_cache, new_chunk):
    """Append new KV chunk to CPU-side KV cache, keeping result in pinned memory."""
    return tuple(
        (torch.cat([old_k, new_k], dim=-2).contiguous().pin_memory(),
         torch.cat([old_v, new_v], dim=-2).contiguous().pin_memory())
        for (old_k, old_v), (new_k, new_v) in zip(cpu_kv_cache, new_chunk)
    )


def build_kv_cache(kv_layers):
    """Reconstruct a DynamicCache from a list of (key, value) tensor pairs."""
    cache = DynamicCache()
    for layer_idx, (k, v) in enumerate(kv_layers):
        cache.update(k, v, layer_idx)
    return cache

def run_dejavu_benchmark():
    microbatches_count = [16]
    n_tokens = [512]
    results = []

    total_vram_mb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 2)

    for D in microbatches_count:
        for N in n_tokens:

            prompts = LONG_PROMPTS[:D]

            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()

            total_start = time.perf_counter()

            # Only profile the D=16, N=512 case
            use_profiler = (D == 16 and N == 512)

            with torch.no_grad():

                # PREFILL
                kv_caches = []
                next_tokens = []
                for i in range(D):
                    inputs = tokenizer(prompts[i], return_tensors="pt").to(device)
                    outputs = model(input_ids=inputs.input_ids, use_cache=True)
                    kv_caches.append(outputs.past_key_values)
                    next_tokens.append(torch.argmax(outputs.logits[:, -1, :], dim=1).unsqueeze(1))

                # Offload prefill KV to pinned CPU memory
                cpu_kv_caches = [offload_kv_to_cpu(kv) for kv in kv_caches]
                del kv_caches
                torch.cuda.empty_cache()

                prefetch_stream = torch.cuda.Stream()
                offload_stream = torch.cuda.Stream()

                # Prefetch first microbatch before loop
                with torch.cuda.stream(prefetch_stream):
                    prefetched = [(k.to(device, non_blocking=True), v.to(device, non_blocking=True))
                                  for k, v in cpu_kv_caches[0]]

                # Timing accumulators
                t_sync_prefetch = 0.0
                t_sync_offload  = 0.0
                t_kick_prefetch = 0.0
                t_build_cache   = 0.0
                t_forward       = 0.0
                t_kick_offload  = 0.0
                t_argmax        = 0.0

                def run_decode_loop(profiler=None):
                    nonlocal prefetched
                    nonlocal t_sync_prefetch, t_sync_offload, t_kick_prefetch
                    nonlocal t_build_cache, t_forward, t_kick_offload, t_argmax

                    for step in range(N - 1):
                        for i in range(D):
                            next_i = (i + 1) % D

                            t0 = time.perf_counter()
                            prefetch_stream.synchronize()
                            t_sync_prefetch += time.perf_counter() - t0

                            t0 = time.perf_counter()
                            offload_stream.synchronize()
                            t_sync_offload += time.perf_counter() - t0

                            gpu_layers = prefetched

                            t0 = time.perf_counter()
                            with torch.cuda.stream(prefetch_stream):
                                prefetched = [(k.to(device, non_blocking=True), v.to(device, non_blocking=True))
                                              for k, v in cpu_kv_caches[next_i]]
                            t_kick_prefetch += time.perf_counter() - t0

                            t0 = time.perf_counter()
                            gpu_cache = build_kv_cache(gpu_layers)
                            old_seq_len = gpu_cache.get_seq_length()
                            t_build_cache += time.perf_counter() - t0

                            t0 = time.perf_counter()
                            with torch.profiler.record_function("forward_pass"):
                                outputs = model(
                                    input_ids=next_tokens[i],
                                    past_key_values=gpu_cache,
                                    use_cache=True
                                )
                            torch.cuda.synchronize()
                            t_forward += time.perf_counter() - t0

                            t0 = time.perf_counter()
                            with torch.profiler.record_function("offload"):
                                with torch.cuda.stream(offload_stream):
                                    new_chunk = extract_new_kv_chunk(outputs.past_key_values, old_seq_len)
                                    cpu_kv_caches[i] = append_kv_chunk(cpu_kv_caches[i], new_chunk)
                            t_kick_offload += time.perf_counter() - t0

                            t0 = time.perf_counter()
                            next_tokens[i] = torch.argmax(outputs.logits[:, -1, :], dim=1).unsqueeze(1)
                            t_argmax += time.perf_counter() - t0

                            if profiler is not None:
                                profiler.step()

                decode_start = time.perf_counter()

                if use_profiler:
                    with torch.profiler.profile(
                        activities=[
                            torch.profiler.ProfilerActivity.CPU,
                            torch.profiler.ProfilerActivity.CUDA,
                        ],
                        schedule=torch.profiler.schedule(
                            wait=1,    # skip first step
                            warmup=1,  # warmup one step
                            active=3,  # record 3 steps
                            repeat=1
                        ),
                        on_trace_ready=torch.profiler.tensorboard_trace_handler('./profiler_logs/dejavu'),
                        record_shapes=True,
                        with_stack=False
                    ) as prof:
                        run_decode_loop(profiler=prof)
                else:
                    run_decode_loop()

                decode_end = time.perf_counter()
                decode_total = decode_end - decode_start
                total_steps = (N - 1) * D

                print(f"\nD={D}, N={N}, Total decode steps: {total_steps}")
                print(f"Decode total:          {decode_total:.3f}s")
                print(f"\n--- Decode breakdown ---")
                print(f"Sync prefetch (wait):  {t_sync_prefetch:.3f}s  ({100*t_sync_prefetch/decode_total:.1f}%)")
                print(f"Sync offload (wait):   {t_sync_offload:.3f}s  ({100*t_sync_offload/decode_total:.1f}%)")
                print(f"Kick prefetch:         {t_kick_prefetch:.3f}s  ({100*t_kick_prefetch/decode_total:.1f}%)")
                print(f"Build cache:           {t_build_cache:.3f}s  ({100*t_build_cache/decode_total:.1f}%)")
                print(f"Forward pass:          {t_forward:.3f}s  ({100*t_forward/decode_total:.1f}%)")
                print(f"Kick offload:          {t_kick_offload:.3f}s  ({100*t_kick_offload/decode_total:.1f}%)")
                print(f"Argmax:                {t_argmax:.3f}s  ({100*t_argmax/decode_total:.1f}%)")

                if use_profiler:
                    print("\n--- torch.profiler top ops ---")
                    print(prof.key_averages().table(
                        sort_by="cuda_time_total",
                        row_limit=15
                    ))

                torch.cuda.empty_cache()

            total_end = time.perf_counter()

            latency = total_end - total_start
            throughput = (D * N) / latency
            peak_vram_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
            vram_util = (peak_vram_mb / total_vram_mb) * 100
            cpu_kv_mb = sum(kv_cache_size_mb(cache) for cache in cpu_kv_caches)

            print(f"\nD={D:<4} N={N:<4} | Latency={latency:.3f}s | Throughput={throughput:.1f} tok/s | "
                  f"Peak VRAM={peak_vram_mb:.1f}MB | VRAM%={vram_util:.1f}% | CPU KV Cache={cpu_kv_mb:.1f}MB")
            print("-" * 80)

            results.append({
                "Microbatches (D)": D,
                "Tokens Generated (N)": N,
                "Latency (s)": latency,
                "Throughput (tok/s)": throughput,
                "Peak VRAM (MB)": peak_vram_mb,
                "VRAM Util (%)": vram_util,
                "CPU KV Cache Total (MB)": cpu_kv_mb
            })

            del next_tokens, cpu_kv_caches
            torch.cuda.empty_cache()

    return pd.DataFrame(results)

run_dejavu_benchmark()


D=16, N=512, Total decode steps: 8176
Decode total:          519.424s

--- Decode breakdown ---
Sync prefetch (wait):  0.188s  (0.0%)
Sync offload (wait):   0.046s  (0.0%)
Kick prefetch:         10.636s  (2.0%)
Build cache:           17.053s  (3.3%)
Forward pass:          296.844s  (57.1%)
Kick offload:          191.402s  (36.8%)
Argmax:                0.980s  (0.2%)

--- torch.profiler top ops ---
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
  

,Microbatches (D),Tokens Generated (N),Latency (s),Throughput (tok/s),Peak VRAM (MB),VRAM Util (%),CPU KV Cache Total (MB)
0,16,512,523.107525,15.66026,14138.238281,34.959836,4585.5
